# Figure 4 Dashboard — Hysteresis Parameter Explorer

This notebook provides **5 interactive dashboards** for exploring how different parameter groups
affect the ABA and stomatal conductance hysteresis curves (Figure 4 from Merilo 2018).

Each dashboard lets you adjust a subset of model parameters using sliders,
then re-runs the drydown (ψ_xyl: 0 → −1.2 MPa) and rewater (ψ_xyl: −1.2 → 0 MPa) simulation.
The resulting curves are plotted alongside the **baseline** (default parameters, gray).

**Dashboards:**
1. **Hydraulics** — tissue resistances, elastic moduli, conductance gain
2. **Vstress → ΔPg curve** — stress-sensing sigmoid shape
3. **ABA Biosynthesis** — synthesis/catabolism kinetics, feedback strength
4. **ABA Signaling** — receptor-kinase-phosphatase cascade
5. **Electrophysiology** — ion channels, pumps, membrane properties

> ⚠️ Each simulation runs ~40 ODE integrations (20 drydown + 20 rewater steps). Expect ~30–60 seconds per run.


In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import io, contextlib, warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import pandas as pd

import panel as pn
pn.extension("bokeh")

print('Imports loaded (matplotlib + Panel).')

## 2. Model Parameters


In [ ]:
def read_parameters_wt_hysteresis():
    """
    Returns a dictionary of model parameters for the coupled hydropassive-
    hydroactive stomatal conductance model — hysteresis scenario.
    
    Key differences from WT transients:
      - φ₁ = 0.06 (reduced stress synthesis)
      - φ₂ = 0.765 (stronger positive feedback → bistability)
      - Shorter equilibration time (24 ks per ψ_xyl step)
    
    The stronger positive feedback combined with reduced stress input creates
    a bistable ABA regime that produces hysteresis during drydown/rewater.
    """
    hyd = {}

    # =========================================================================
    # 1. TISSUE VOLUMES & TURGOR PRESSURES
    # =========================================================================
    # Symplastic water volumes (mmol H₂O / m² leaf area)
    hyd["Vg0"] = 100.0                     # guard cell
    hyd["Vs0"] = 100.0                     # subsidiary cell
    hyd["Vm0"] = 1978.0                    # mesophyll

    hyd["stomatal_density"] = 150e6        # stomata / m² leaf

    # Bulk elastic moduli (MPa) — tissue stiffness
    hyd["epsilong"] = 3.6                  # guard cell
    hyd["epsilons"] = 0.5                  # subsidiary cell
    hyd["epsilonm"] = 4.0                  # mesophyll

    # Initial turgor pressures (MPa)
    hyd["Pg0"] = 2.5                       # guard cell
    hyd["Ps0"] = 1.0                       # subsidiary cell
    hyd["Pm0"] = 2.2                       # mesophyll

    # Initial osmotic pressures (MPa); offset from turgor by 0.1
    hyd["pig0"] = hyd["Pg0"] + 0.1
    hyd["pis0"] = hyd["Ps0"] + 0.1
    hyd["pim0"] = hyd["Pm0"] + 0.1

    # Initial turgor perturbations (MPa)
    hyd["DeltaPim"] = 0.0
    hyd["DeltaPis"] = 0.0
    hyd["DeltaPig"] = 0.0

    # =========================================================================
    # 2. GAS CONSTANTS & HYDRAULIC TRANSPORT
    # =========================================================================
    hyd["R"]    = 8.314                    # universal gas constant (J/mol/K)
    hyd["T"]    = 308.0                    # temperature (K) ≈ 35°C
    hyd["nu_w"] = 1.8e-5                   # molar volume of water (m³/mol)
    hyd["RT_by_nu_by1e6"] = (hyd["R"] * hyd["T"] / hyd["nu_w"]) / 1e6

    # Stomatal conductance equation:  gs = Xi * max(Pg - m*Ps, 0)
    hyd["Xi"] = 295.0                      # conductance gain (mmol/m²/s/MPa)
    hyd["m"]  = 1.71                       # mechanical advantage of subsidiary cells

    # Hydraulic resistances (MPa·s·m²/mmol) — flow = ΔΨ / R
    hyd["Rxm"] = 0.018116                  # xylem → mesophyll
    hyd["Rms"] = 5.0                       # mesophyll → subsidiary cell
    hyd["Rsg"] = 10.0                      # subsidiary cell → guard cell

    # =========================================================================
    # 3. CONDUCTANCE BASELINES & ATMOSPHERIC CONDITIONS
    # =========================================================================
    # Stomatal and cuticular conductance coefficients (mmol/m²/s)
    hyd["gg1"] = 2.9                       # g_g¹ — guard cell (stomatal) pathway
    hyd["ge1"] = 2.9                       # g_e¹ — epidermal (cuticular) pathway
    hyd["gg2"] = 1.0                       # g_g² — guard cell VPD sensitivity
    hyd["ge2"] = 10.0                      # g_e² — epidermal VPD sensitivity

    # Saturation vapor pressure and atmospheric pressure
    hyd["psat"]      = 0.00563             # saturation vapor pressure (MPa) at T
    hyd["patm"]      = 0.101               # atmospheric pressure (MPa)
    hyd["psat_patm"] = hyd["psat"] / hyd["patm"]

    # =========================================================================
    # 4. HUMIDITY PROTOCOL (constant RH, stepped ψ_xyl)
    # =========================================================================
    hyd["RH1"] = 80.0                      # constant RH during hysteresis (%)
    hyd["RH2"] = 60.0                      # VPD step for equilibration
    hyd["VPD1"] = hyd["psat"] * (1 - hyd["RH1"] / 100)
    hyd["VPD2"] = hyd["psat"] * (1 - hyd["RH2"] / 100)
    hyd["psix_at_RH_step"] = -0.1          # initial xylem water potential (MPa)

    # =========================================================================
    # 5. ABA BIOSYNTHESIS & CATABOLISM
    #    Autoregulatory motif: NCED (positive fb) + CYP707A/A8H (negative fb)
    #    HYSTERESIS REGIME: φ₂ = 0.765 → stronger positive feedback → bistability
    # =========================================================================
    # Genetic pre-factors (1.0 = wild-type; set to 0 for knockouts)
    hyd["thetaABA"] = 1.0                  # θ_NCED — synthesis (NCED3/5)
    hyd["thetaA8H"] = 1.0                  # θ_A8H  — catabolism (CYP707A)

    hyd["n"]    = 3                        # Hill coefficient (stress cooperativity)
    hyd["kcat"] = 0.25                     # γ_A: A8H catalytic rate (1/s)
    hyd["K2"]   = 2000.0                   # K_d: A8H Michaelis constant (nM)
    hyd["gammac"] = 32 * 9.6e-5            # γ_H: A8H protein decay rate (1/s) ≈ 3.072e-3

    # Half-saturation ratios: K₊ = λ₁·K_d,  K₋ = λ₃·K_d
    hyd["lambda1"] = 0.78
    hyd["lambda3"] = 0.78
    hyd["K1"] = hyd["lambda1"] * hyd["K2"]  # K₊ (nM) — NCED half-saturation
    hyd["K3"] = hyd["lambda3"] * hyd["K2"]  # K₋ (nM) — CYP707A half-saturation

    # Dimensionless rate ratios (φ) — hysteresis regime
    hyd["phi1"] = 0.06                     # reduced stress synthesis
    hyd["phi2"] = 0.765                    # stronger positive feedback (bistability)
    hyd["phi3"] = 20.0                     # standard catabolism scaling

    # Derived maximal rates (nM/s)
    hyd["Vminus"]  = (hyd["K2"] * hyd["gammac"]**2 * hyd["phi3"]) / hyd["kcat"]
    hyd["Vplus"]   = hyd["phi2"] * hyd["kcat"] * hyd["Vminus"] / hyd["gammac"]
    hyd["Vstress"] = hyd["phi1"] * hyd["kcat"] * hyd["Vminus"] / hyd["gammac"]

    # =========================================================================
    # 6. ABA SIGNALING CASCADE
    #    Bicyclic futile cycle: MAP3K → OST1 → SLAC1
    #    with PP2C dephosphorylation, inhibited by ABA:PYR complex
    # =========================================================================
    # Total protein concentrations (nM)
    hyd["OST1T"]  = 1e3
    hyd["SLAC1T"] = 1e3
    hyd["MAP3KT"] = 1e3
    hyd["PP2CT"]  = 1e3
    hyd["PYRT"]   = 1e3

    # ABA-receptor binding
    hyd["KD"] = 1000.0                     # ABA–PYR dissociation constant (nM)
    hyd["KI"] = 30.0                       # PYR–PP2C inhibition constant (nM)

    # Phosphorylation / dephosphorylation rates (1/s)
    hyd["k1"] = 0.03                       # MAP3K → OST1 (phosphorylation)
    hyd["k2"] = 0.032772                   # PP2C ⊣ OST1 (dephosphorylation)
    hyd["k3"] = 3.75                       # OST1 → SLAC1 (phosphorylation)
    hyd["k4"] = 0.1386                     # PP2C ⊣ SLAC1 (dephosphorylation)

    # Michaelis constants for the four enzymatic steps (nM)
    hyd["Km1"] = 2.5e3
    hyd["Km2"] = 0.0897e3
    hyd["Km3"] = 18.93e3
    hyd["Km4"] = 0.597e3

    # =========================================================================
    # 7. GUARD CELL ELECTROPHYSIOLOGY
    #    GHK ion fluxes through SLAC1 (Cl⁻), GORK (K⁺ out), KAT1 (K⁺ in),
    #    and H⁺-ATPase proton pump
    # =========================================================================
    # Membrane properties
    hyd["Cm"]   = 1e-2                     # membrane capacitance (F/m²)
    hyd["area"] = 20e-10                   # guard cell membrane area (m²)
    hyd["vol_in"]  = hyd["Vg0"] * hyd["nu_w"] * 1e-3 / hyd["stomatal_density"]  # intracellular volume (m³)
    hyd["vol_out"] = 65e-15                # apoplastic volume (m³)
    hyd["F"] = 96485.3329                  # Faraday constant (C/mol)

    # GORK voltage gating
    hyd["delta_GORK"]         = 2.0        # voltage sensitivity parameter
    hyd["V1by2_Kconc_GORK"]   = 150.0      # half-activation K⁺ concentration (mM)

    # KAT1 voltage gating
    hyd["delta_KAT1"]  = 1.6              # voltage sensitivity parameter
    hyd["V1by2_KAT1"]  = -170e-3          # half-activation voltage (V)

    hyd["otherIons"] = 0.0                 # background ion contribution

    # Initial ion concentrations (mM)
    hyd["Clin0"]  = 260.0                  # Cl⁻ intracellular
    hyd["Clout0"] = 22.0                   # Cl⁻ apoplastic
    hyd["Kin0"]   = 210.0                  # K⁺  intracellular
    hyd["Kout0"]  = 20.0                   # K⁺  apoplastic

    # Ion valences
    hyd["zCl"] = -1
    hyd["zK"]  = 1

    # Membrane permeabilities (m/s)
    hyd["PK"]  = 5e-6                      # K⁺ permeability
    hyd["PCl"] = 1e-7                      # Cl⁻ permeability

    # Channel number scaling (relative to baseline)
    hyd["NGORK"]  = 1.0
    hyd["NKAT1"]  = 1.0
    hyd["NSLAC1"] = 1.0

    # Proton concentrations (derived from pH)
    hyd["pHin0"]  = 7.2
    hyd["pHout0"] = 6.5
    hyd["Hin0"]   = 10**(-hyd["pHin0"])  * 1e3   # intracellular H⁺ (mM)
    hyd["Hout0"]  = 10**(-hyd["pHout0"]) * 1e3   # apoplastic H⁺ (mM)
    hyd["zH"] = 1

    # H⁺-ATPase proton pump
    hyd["DeltaGATP"]      = -26000.0       # free energy of ATP hydrolysis (J/mol)
    hyd["E0pump"]         = 1.1033e6       # pump rate constant
    hyd["k1pump"]         = 0.55           # pump kinetic parameter 1
    hyd["k2pump"]         = 2.58e-4        # pump kinetic parameter 2
    hyd["V0Cl"]           = 9.25e4         # SLAC1 Cl⁻ flux scaling
    hyd["ABA_Hpump_Khalf"] = 50.0          # ABA half-inhibition of H⁺-pump (nM)
    hyd["ABA_Hpump_n"]    = 1.0            # Hill coefficient for ABA pump inhibition

    # =========================================================================
    # 8. SIMULATION TIMING (equilibration per ψ_xyl step)
    # =========================================================================
    hyd["tstart"] = 0.0                    # simulation start (s)
    hyd["tstep1"] = 24000.0                # equilibration time per step (s) = 400 min
    hyd["tend"]   = hyd["tstep1"] * 2      # total run per step (s)   = 800 min

    return hyd

print("Parameter function defined.")

## 3. Helper Functions


In [ ]:
# =============================================================================
# HELPER FUNCTIONS
# Organized by module: Environmental → Hydraulics → ABA → Signaling →
#                      Ion Channels → Membrane Transport
# =============================================================================

# ---- Environmental forcing (hysteresis: constant RH, stepped ψ_xyl) --------

def f_RH_hyst(t, hyd):
    """Relative humidity held constant at RH1 throughout hysteresis protocol."""
    return np.where(np.asarray(t) <= hyd["tstep1"], hyd["RH1"], hyd["RH1"])

def f_VPD_hyst(t, hyd):
    """Vapor pressure deficit step (used for equilibration phase)."""
    return np.where(np.asarray(t) <= hyd["tstep1"], hyd["VPD1"], hyd["VPD2"])

def f_psix_hyst(t, hyd):
    """Xylem water potential protocol for hysteresis.

    Initial equilibration at ψ_xyl = -0.3 MPa for t ≤ tstep1,
    then steps to psix_input (the current drydown/rewater value).
    """
    return np.where(np.asarray(t, dtype=float) <= hyd["tstep1"], -0.3, hyd["psix_input"])

# ---- Hydraulic transport ----------------------------------------------------

def f_Rox(psix):
    """Overall xylem resistance as a function of xylem water potential.

    Empirical sigmoidal fit capturing vulnerability-curve–driven loss of
    xylem hydraulic conductance under drought.
    """
    a, b, c, k = 0.3664, 15.61, -0.7684, 2.958
    return 1.0 / (a + (k - a) / (1.0 + np.exp(-b * (psix - c))))

def f_P_from_psi(psi, epsilon, pi0, P0, DeltaPi_MPa):
    """Convert water potential perturbation (ψ) to turgor pressure (P).

    Uses the linearized relation:  P = P0 + ψ / (1 + π0/ε)
    Turgor is clamped to ≥ 0 (no negative turgor).
    """
    return np.maximum(P0 + (psi - DeltaPi_MPa) / (1.0 + pi0/epsilon), 0.0)

def f_SSC_apoplast_potential(psim, psig, psis, gs, gmH2O, pa, hyd, geH2O, ggH2O):
    """Steady-state substomatal cavity (SSC) apoplast water potential.

    Balances water inflow from mesophyll, subsidiary, and guard cells against
    evaporative loss through stomata.
    """
    return (psim*gmH2O + psis*geH2O + psig*ggH2O
            - gs*hyd["RT_by_nu_by1e6"]*(hyd["psat"]-pa)/hyd["psat"])             / (gmH2O + geH2O + ggH2O + gs)

# ---- ABA biosynthesis & catabolism ------------------------------------------

def f_Vstress(DeltaPg, hyd):
    """Stress-responsive ABA synthesis rate as a Hill function of turgor loss.

    V_stress(ΔPg) = V_stress · |ΔPg|^n / (K_half + |ΔPg|^n)

    Uses custom vstress_exponent and vstress_khalf if provided via
    dashboard overrides, otherwise defaults to n=3.25 and K_half=1.4.
    """
    abs_dpg = np.abs(DeltaPg)
    n_hill = hyd.get("vstress_exponent", 3.25)
    k_half = hyd.get("vstress_khalf", 1.4)
    return np.abs(hyd["Vstress"] * abs_dpg**n_hill / (k_half + abs_dpg**n_hill))

# ---- ABA signaling cascade -------------------------------------------------

def f_solve_ABA_PYR_quadratic(ABA, hyd):
    """Solve for free PYR receptor concentration given [ABA].

    ABA binds PYR receptors; the ABA:PYR complex inhibits PP2C. Solves
    the quadratic mass-balance for [PYR_free].
    """
    theta_PYR = 1.0 + hyd["KD"] / ABA
    b = hyd["KI"] + (hyd["PP2CT"] - hyd["PYRT"]) / theta_PYR
    c = -hyd["KI"] * hyd["PYRT"] / theta_PYR
    return (-b + np.sqrt(b**2 - 4.0*c)) / 2.0

def f_SLAC1_conc_non_dimensional(Oo, So, hyd):
    """Unphosphorylated SLAC1 fraction from conservation law."""
    K4p   = hyd["K4_nd"] * (1.0 + Oo/hyd["K2_nd"])
    term3 = (hyd["PP2CT"]/hyd["SLAC1T"]) * So / (K4p + So)
    return (1.0 - So - term3) / (1.0 + (hyd["OST1T"]/hyd["SLAC1T"]) * (Oo/hyd["K3_nd"]))

def f_OST1_roots_from_quadratic_non_dimensional(Oo, So, hyd):
    """Unphosphorylated OST1 fraction from conservation law.

    Solves the quadratic from OST1 total conservation. Returns positive root.
    """
    S     = f_SLAC1_conc_non_dimensional(Oo, So, hyd)
    theta = 1.0 - Oo * (1.0 + S/hyd["K3_nd"])
    b     = hyd["epsilon1"] + hyd["K1_nd"] - theta
    c     = -theta * hyd["K1_nd"]
    return max((-b + np.sqrt(b**2 - 4.0*c)) / 2.0, 0.0)

# ---- Ion channel gating ----------------------------------------------------

def f_p_open_KAT1_channel(Vmemb, V1by2_KAT1, delta_KAT1, hyd):
    """KAT1 inward-rectifying K⁺ channel open probability (Boltzmann gating)."""
    return 1.0 / (1.0 + np.exp((Vmemb - V1by2_KAT1)*(delta_KAT1*hyd["F"]/(hyd["R"]*hyd["T"]))))

def f_V1by2_GORK(Kout, hyd):
    """GORK half-activation voltage (depends on external K⁺ via Nernst shift)."""
    return 1e-3 * (hyd["F"]/(hyd["R"]*hyd["T"])) * np.log(Kout/hyd["V1by2_Kconc_GORK"])

def f_p_open_GORK_channel(Vmemb, V1by2_GORK, delta_GORK, hyd):
    """GORK outward-rectifying K⁺ channel open probability (Boltzmann gating)."""
    return 1.0 / (1.0 + np.exp(-(Vmemb - V1by2_GORK)*(delta_GORK*hyd["F"]/(hyd["R"]*hyd["T"]))))

# ---- Membrane transport (GHK fluxes, mass balance, H⁺-pump) ----------------

def f_GHK_current_flux(Pion, zion, Vmemb, ionConcIn, ionConcOut, hyd):
    """Goldman-Hodgkin-Katz current density for a single ion species."""
    zeta = Vmemb * hyd["F"] / (hyd["R"] * hyd["T"])
    return (Pion * zion**2 * hyd["F"] * zeta
            * (ionConcIn - ionConcOut*np.exp(-zion*zeta))
            / (1.0 - np.exp(-zion*zeta)))

def f_mass_balance(ionIn, ionIn0, ionOut0, hyd):
    """Apoplastic ion concentration from total ion conservation."""
    Q = ionOut0*hyd["vol_out"] + ionIn0*hyd["vol_in"]
    return (Q - ionIn*hyd["vol_in"]) / hyd["vol_out"]

def f_p_on_Hpump(ABA, hyd):
    """Fraction of H⁺-ATPase pumps active (ABA inhibits the pump).

    Hill inhibition: p_on = 1 / (1 + ([ABA]/K_half)^n)
    """
    return 1.0 / (1.0 + (ABA/hyd["ABA_Hpump_Khalf"])**hyd["ABA_Hpump_n"])

def f_HPump_current(p_on_Hpump, Hin, Hout, Vmemb, hyd):
    """H⁺-ATPase electrogenic pump current (two-state kinetic model)."""
    u       = Vmemb * hyd["F"] / (hyd["R"] * hyd["T"])
    kplus1  = hyd["k1pump"] * Hin
    kplus2  = hyd["k2pump"] * u / (1.0 - np.exp(-u))
    kminus1 = hyd["k1pump"] * np.exp(hyd["DeltaGATP"] / (hyd["R"]*hyd["T"]))
    kminus2 = hyd["k2pump"] * Hout * u * np.exp(-u) / (1.0 - np.exp(-u))
    cyc = (kplus1*kplus2 - kminus1*kminus2) / (kplus1 + kplus2 + kminus1 + kminus2)
    return p_on_Hpump * hyd["E0pump"] * cyc

def f_2HClSymport_current(p_on_Hpump, Hin, Hout, Clin, Clout, Vmemb, hyd):
    """2H⁺/Cl⁻ symporter current (secondary active Cl⁻ uptake)."""
    u = Vmemb * hyd["F"] / (hyd["R"] * hyd["T"])
    return p_on_Hpump * hyd["V0Cl"] * (
        Clin*Hin**2*u - Clout*Hout**2*u*np.exp(-u)) / (1.0 - np.exp(-u))

print("Helper functions defined.")

## 4. ODE System


In [ ]:
def fullModel_DeltaPg_ode_hyst(t, y, hyd):
    """
    Full 10-variable ODE system for the coupled HP-HA stomatal model.
    Hysteresis variant using stepped ψ_xyl protocol with constant RH.

    State vector y = [ψ_m, ψ_s, ψ_g, ABA, A8H, Oo, So, V_memb, Cl_in, K_in]

    Modules:
      1. Hydraulics      — tissue water potentials → turgor pressures → gs
      2. ABA biosynthesis — stress-sensing + autoregulatory motif → [ABA], [A8H]
      3. ABA signaling    — receptor–kinase–phosphatase cascade → OST1*, SLAC1*
      4. Electrophysiology — ion channels + pump → membrane potential + ion fluxes
    """
    # Unpack state variables
    psim, psis, psig, ABA, A8H, Oo, So, Vmemb, Clin, Kin = y

    # ---- Environmental forcing (constant RH, stepped ψ_xyl) ----
    RH   = float(f_RH_hyst(t, hyd))
    psix = float(f_psix_hyst(t, hyd))
    pa   = hyd["psat"] * RH / 100.0            # ambient vapor pressure (MPa)

    # =========================================================================
    # MODULE 1: HYDRAULICS
    # =========================================================================
    pm = hyd["psat"] * (1.0 + psim/hyd["RT_by_nu_by1e6"])
    ps = hyd["psat"] * (1.0 + psis/hyd["RT_by_nu_by1e6"])
    pg = hyd["psat"] * (1.0 + psig/hyd["RT_by_nu_by1e6"])

    Rg    = (1.0/hyd["gg1"]) * hyd["RT_by_nu_by1e6"] * hyd["patm"] / hyd["psat"]
    Re    = (1.0/hyd["ge1"]) * hyd["RT_by_nu_by1e6"] * hyd["patm"] / hyd["psat"]
    Req1  = Re / (1.0 + Re/(hyd["Rsg"] + Rg))
    Rox   = f_Rox(psix)
    Rm    = (Rox - hyd["Rxm"]) / (1.0 + (Rox - hyd["Rxm"])/(hyd["Rms"] + Req1))
    gmH2O = (1.0/Rm) * hyd["RT_by_nu_by1e6"] * hyd["patm"] / hyd["psat"]

    DeltaPig_MPa = (hyd["Kin0"]+hyd["Clin0"]-Kin-Clin+hyd["otherIons"]) * hyd["R"]*hyd["T"] / 1e6
    Pg = f_P_from_psi(psig, hyd["epsilong"], hyd["pig0"], hyd["Pg0"], DeltaPig_MPa)
    Ps = f_P_from_psi(psis, hyd["epsilons"], hyd["pis0"], hyd["Ps0"], 0.0)
    gs = max(hyd["Xi"] * (Pg - hyd["m"]*Ps), 0.0)

    psii = f_SSC_apoplast_potential(psim, psig, psis, gs, gmH2O, pa, hyd, hyd["ge1"], hyd["gg1"])
    p_i  = hyd["psat"] * (1.0 + psii/hyd["RT_by_nu_by1e6"])

    Em  = gmH2O*      (pm - p_i) / hyd["patm"]
    Es1 = hyd["ge1"] *(ps - p_i) / hyd["patm"]
    Eg1 = hyd["gg1"] *(pg - p_i) / hyd["patm"]
    Es2 = hyd["ge2"] *(ps - pa)  / hyd["patm"]
    Eg2 = hyd["gg2"] *(pg - pa)  / hyd["patm"]

    Cm_c = hyd["Vm0"]/(hyd["epsilonm"]+hyd["pim0"])
    Cs_c = hyd["Vs0"]/(hyd["epsilons"]+hyd["pis0"])
    Cg_c = hyd["Vg0"]/(hyd["epsilong"]+hyd["pig0"])

    dpsimdt = (1/Cm_c)*(psix/hyd["Rxm"]+psim*(-1/hyd["Rxm"]-1/hyd["Rms"])+psis/hyd["Rms"]-Em)
    dpsisdt = (1/Cs_c)*(psim/hyd["Rms"]+psis*(-1/hyd["Rms"]-1/hyd["Rsg"])+psig/hyd["Rsg"]-Es1-Es2)
    dpsigdt = (1/Cg_c)*(psis/hyd["Rsg"]+psig*(-1/hyd["Rsg"])-Eg1-Eg2)

    # =========================================================================
    # MODULE 2: ABA BIOSYNTHESIS & CATABOLISM
    # =========================================================================
    DeltaPg_val = Pg - hyd["Pg0"]
    Vstress_val = f_Vstress(DeltaPg_val, hyd)

    dABAdt = (hyd["thetaABA"]
              * (Vstress_val + hyd["Vplus"]*ABA**hyd["n"]/(hyd["K1"]**hyd["n"]+ABA**hyd["n"]))
              - hyd["kcat"]*A8H*ABA/hyd["K2"])

    dA8Hdt = (hyd["thetaA8H"]*hyd["Vminus"]*ABA/(hyd["K3"]+ABA)
              - hyd["gammac"]*A8H)

    # =========================================================================
    # MODULE 3: ABA SIGNALING CASCADE
    # =========================================================================
    K1_nd = hyd["Km1"]/hyd["OST1T"]; K2_nd = hyd["Km2"]/hyd["OST1T"]
    K3_nd = hyd["Km3"]/hyd["SLAC1T"]; K4_nd = hyd["Km4"]/hyd["SLAC1T"]
    V1 = hyd["k1"]*hyd["MAP3KT"]; V2 = hyd["k2"]*hyd["PP2CT"]
    V3 = hyd["k3"]*hyd["OST1T"];  V4 = hyd["k4"]*hyd["PP2CT"]
    epsilon1 = hyd["MAP3KT"]/hyd["OST1T"]
    hx = {**hyd, "K1_nd":K1_nd, "K2_nd":K2_nd, "K3_nd":K3_nd, "K4_nd":K4_nd, "epsilon1":epsilon1}

    ABA_PYR = f_solve_ABA_PYR_quadratic(ABA, hyd)
    V2p = V2/(1.0+ABA_PYR/hyd["KI"]); V4p = V4/(1.0+ABA_PYR/hyd["KI"])
    K2prime = K2_nd*(1.0+So/K4_nd); K4prime = K4_nd*(1.0+Oo/K2_nd)

    O = f_OST1_roots_from_quadratic_non_dimensional(Oo, So, hx)
    dOodt = (V2p/hyd["OST1T"])*((V1/V2p)*O/(K1_nd+O) - Oo/(K2prime+Oo))

    S = f_SLAC1_conc_non_dimensional(Oo, So, hx)
    dSodt = (V4p/hyd["SLAC1T"])*((V3/V4p)*Oo*S/K3_nd - So/(K4prime+So))

    # =========================================================================
    # MODULE 4: GUARD CELL ELECTROPHYSIOLOGY
    # =========================================================================
    Clout = f_mass_balance(Clin, hyd["Clin0"], hyd["Clout0"], hyd)
    Kout  = f_mass_balance(Kin,  hyd["Kin0"],  hyd["Kout0"],  hyd)

    IGHK_Cl   = f_GHK_current_flux(hyd["PCl"], hyd["zCl"], Vmemb, Clin, Clout, hyd)
    ICl_SLAC1 = (hyd["SLAC1T"]/1e3)*So*IGHK_Cl

    p_KAT1  = f_p_open_KAT1_channel(Vmemb, hyd["V1by2_KAT1"], hyd["delta_KAT1"], hyd)
    IGHK_K  = f_GHK_current_flux(hyd["PK"], hyd["zK"], Vmemb, Kin, Kout, hyd)
    IK_KAT1 = hyd["NKAT1"]*p_KAT1*IGHK_K

    V12G    = f_V1by2_GORK(Kout, hyd)
    p_GORK  = f_p_open_GORK_channel(Vmemb, V12G, hyd["delta_GORK"], hyd)
    IK_GORK = hyd["NGORK"]*p_GORK*IGHK_K

    p_pump  = f_p_on_Hpump(ABA, hyd)
    IH_pump = f_HPump_current(p_pump, hyd["Hin0"], hyd["Hout0"], Vmemb, hyd)
    ICl_sym = f_2HClSymport_current(1.0, hyd["Hin0"], hyd["Hout0"], Clin, Clout, Vmemb, hyd)

    dVmembdt = -(1/hyd["Cm"])*(IH_pump+ICl_SLAC1+IK_KAT1+IK_GORK+ICl_sym)
    dClindt  = -(ICl_SLAC1-ICl_sym)*hyd["zCl"]*hyd["area"]/(hyd["F"]*hyd["vol_in"])
    dKindt   = -(IK_GORK+IK_KAT1)*hyd["zK"]*hyd["area"]/(hyd["F"]*hyd["vol_in"])

    return [dpsimdt, dpsisdt, dpsigdt, dABAdt, dA8Hdt,
            dOodt, dSodt, dVmembdt, dClindt, dKindt]


def evaluate_output_hyst(t, y, hyd):
    """Post-process ODE solution for hysteresis.

    Returns dict with ABA and gs arrays (used for extracting steady-state values).
    """
    psig = y[:,2]; psis = y[:,1]; Clin = y[:,8]; Kin = y[:,9]
    DeltaPig_MPa = (hyd["Kin0"]+hyd["Clin0"]-Kin-Clin+hyd["otherIons"]) * hyd["R"]*hyd["T"] / 1e6
    Pg = f_P_from_psi(psig, hyd["epsilong"], hyd["pig0"], hyd["Pg0"], DeltaPig_MPa)
    Ps = f_P_from_psi(psis, hyd["epsilons"], hyd["pis0"], hyd["Ps0"], 0.0)
    gs = np.maximum(hyd["Xi"]*(Pg - hyd["m"]*Ps), 0.0)
    return dict(ABA=y[:,3], gs=gs)

print("ODE and evaluation functions defined.")

## 5. Hysteresis Simulation Engine


In [ ]:
N_STEPS = 20  # steps per direction (reduced from 50 for speed)

def run_hysteresis(hyd_overrides=None, n_steps=N_STEPS, progress_label=None):
    """Run full drydown + rewater hysteresis loop."""
    psix_dd = np.linspace(0, -1.2, n_steps)
    psix_rw = np.linspace(-1.2, 0, n_steps)
    ABA_dd = np.zeros(n_steps); gs_dd = np.zeros(n_steps)
    ABA_rw = np.zeros(n_steps); gs_rw = np.zeros(n_steps)
    tspan = np.arange(0, 48030, 30.0)
    y0 = [0.0, 0.0, 0.0, 1e-3, 1e-3, 0.0, 0.0, -180e-3, 260.0, 210.0]

    for phase, psix_arr, ABA_out, gs_out, phase_name in [
        (0, psix_dd, ABA_dd, gs_dd, "Drydown"),
        (1, psix_rw, ABA_rw, gs_rw, "Rewater")]:

        for i, psi_val in enumerate(psix_arr):
            with contextlib.redirect_stdout(io.StringIO()):
                hyd_i = read_parameters_wt_hysteresis()
            if hyd_overrides:
                # Pop direct ABA rates
                direct_rates = {}
                for k in ("Vstress", "Vplus", "Vminus"):
                    if k in hyd_overrides:
                        direct_rates[k] = hyd_overrides[k]

                # Pop combined K+=K- override
                K_pm_val = hyd_overrides.get("K_pm", None)

                for k, v in hyd_overrides.items():
                    if k not in ("Vstress", "Vplus", "Vminus", "K_pm"):
                        hyd_i[k] = v

                # Recompute derived quantities
                hyd_i["K1"] = hyd_i["lambda1"] * hyd_i["K2"]
                hyd_i["K3"] = hyd_i["lambda3"] * hyd_i["K2"]

                # Apply combined K+=K-
                if K_pm_val is not None:
                    hyd_i["K1"] = K_pm_val
                    hyd_i["K3"] = K_pm_val

                hyd_i["pig0"] = hyd_i["Pg0"] + 0.1
                hyd_i["pis0"] = hyd_i["Ps0"] + 0.1
                hyd_i["pim0"] = hyd_i["Pm0"] + 0.1
                hyd_i["vol_in"] = hyd_i["Vg0"] * hyd_i["nu_w"] * 1e-3 / hyd_i["stomatal_density"]

                if direct_rates:
                    hyd_i["Vstress"] = direct_rates.get("Vstress", hyd_i["Vstress"])
                    hyd_i["Vplus"]   = direct_rates.get("Vplus",   hyd_i["Vplus"])
                    hyd_i["Vminus"]  = direct_rates.get("Vminus",  hyd_i["Vminus"])
                else:
                    hyd_i["Vminus"] = (hyd_i["K2"] * hyd_i["gammac"]**2 * hyd_i["phi3"]) / hyd_i["kcat"]
                    hyd_i["Vplus"]  = hyd_i["phi2"] * hyd_i["kcat"] * hyd_i["Vminus"] / hyd_i["gammac"]
                    hyd_i["Vstress"]= hyd_i["phi1"] * hyd_i["kcat"] * hyd_i["Vminus"] / hyd_i["gammac"]

            hyd_i["psix_input"] = psi_val

            sol = solve_ivp(
                lambda t, y, h=hyd_i: fullModel_DeltaPg_ode_hyst(t, y, h),
                [hyd_i["tstart"], hyd_i["tend"]],
                y0, method="BDF", t_eval=tspan, rtol=1e-6, atol=1e-9)
            out = evaluate_output_hyst(sol.t, sol.y.T, hyd_i)
            gs_out[i] = out["gs"][-1]; ABA_out[i] = out["ABA"][-1]
            y0 = list(sol.y[:, -1])
            y0[8] = hyd_i["Clin0"]; y0[9] = hyd_i["Kin0"]
            if progress_label:
                progress_label.value = f"{phase_name} step {i+1}/{n_steps}  psi_xyl={psi_val:.2f}"

    return psix_dd, ABA_dd, gs_dd, psix_rw, ABA_rw, gs_rw

print(f"Hysteresis engine ready ({N_STEPS} steps per direction).")


## 6. Baseline Hysteresis (Default Parameters)


In [ ]:
print("Running baseline hysteresis (this takes ~30-60 seconds)...")
baseline = run_hysteresis(hyd_overrides=None)
psix_dd_base, ABA_dd_base, gs_dd_base, psix_rw_base, ABA_rw_base, gs_rw_base = baseline
print("Baseline complete!")


## Figure 4: Steady-State Drydown / Rewater Hysteresis

Publication-quality plots showing hysteresis in the ABA–stomatal conductance
relationship during progressive soil drydown followed by rewatering. At each
xylem water potential the model is run to steady state; the final state feeds
the next step (quasi-static experiment).

In [ ]:
# ── Figure 4: ABA & gs Hysteresis (side by side) ─────────────────────────────
fig_hysteresis, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# ── Panel (a): ABA hysteresis ────────────────────────────────────────────────
ax1.plot(psix_dd_base, ABA_dd_base, 'r-d', linewidth=2.5, markersize=8,
         markerfacecolor='#e74c3c', markeredgecolor='#c0392b', label='Drydown')
ax1.plot(psix_rw_base, ABA_rw_base, 'b-d', linewidth=2.5, markersize=8,
         markerfacecolor='#3498db', markeredgecolor='#2980b9', label='Rewater')
ax1.set_xlabel(r'$\psi_{xyl}$ (MPa)', fontsize=16)
ax1.set_ylabel('[ABA] (nM)', fontsize=16)
ax1.set_xticks(np.arange(-1.5, 0.1, 0.3))
ax1.tick_params(axis='both', which='major', labelsize=13, width=1.2, length=6)
ax1.tick_params(axis='both', which='minor', width=0.8, length=3)
ax1.minorticks_on()
ax1.grid(True, which='major', alpha=0.3, linewidth=0.8)
ax1.grid(True, which='minor', alpha=0.15, linewidth=0.5)
ax1.legend(fontsize=14, loc='best', framealpha=0.9)
ax1.set_title('Figure 4a: ABA Hysteresis — Drydown vs Rewater', fontsize=16, fontweight='bold')

# ── Panel (b): Stomatal conductance hysteresis ───────────────────────────────
ax2.plot(psix_dd_base, gs_dd_base, 'r-d', linewidth=2.5, markersize=8,
         markerfacecolor='#e74c3c', markeredgecolor='#c0392b', label='Drydown')
ax2.plot(psix_rw_base, gs_rw_base, 'b-d', linewidth=2.5, markersize=8,
         markerfacecolor='#3498db', markeredgecolor='#2980b9', label='Rewater')
ax2.set_xlabel(r'$\psi_{xyl}$ (MPa)', fontsize=16)
ax2.set_ylabel(r'$g_s$ (mmol m$^{-2}$ s$^{-1}$)', fontsize=16)
ax2.set_ylim(0, 200)
ax2.set_xticks(np.arange(-1.5, 0.1, 0.3))
ax2.tick_params(axis='both', which='major', labelsize=13, width=1.2, length=6)
ax2.tick_params(axis='both', which='minor', width=0.8, length=3)
ax2.minorticks_on()
ax2.grid(True, which='major', alpha=0.3, linewidth=0.8)
ax2.grid(True, which='minor', alpha=0.15, linewidth=0.5)
ax2.legend(fontsize=14, loc='best', framealpha=0.9)
ax2.set_title(r'Figure 4b: $g_s$ Hysteresis — Drydown vs Rewater', fontsize=16, fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Dashboard Infrastructure

In [ ]:
from bokeh.plotting import figure
from bokeh.layouts import row as bokeh_row

def make_hysteresis_plot(psix_dd, ABA_dd, gs_dd, psix_rw, ABA_rw, gs_rw, title_suffix):
    """Create a Bokeh row of ABA + gs hysteresis plots."""
    p1 = figure(title=f'ABA Hysteresis \u2014 {title_suffix}',
               x_axis_label='\u03C8_xyl (MPa)', y_axis_label='[ABA] (nM)',
               width=550, height=400)
    p1.line(list(psix_dd_base), list(ABA_dd_base), line_width=1.5, color='gray', alpha=0.5, legend_label='Baseline DD')
    p1.line(list(psix_rw_base), list(ABA_rw_base), line_width=1.5, color='gray', alpha=0.5, line_dash='dashed', legend_label='Baseline RW')
    p1.line(list(psix_dd), list(ABA_dd), line_width=2.5, color='red', legend_label='Drydown')
    p1.scatter(list(psix_dd), list(ABA_dd), size=8, color='#e74c3c', marker='diamond', legend_label='Drydown')
    p1.line(list(psix_rw), list(ABA_rw), line_width=2.5, color='blue', legend_label='Rewater')
    p1.scatter(list(psix_rw), list(ABA_rw), size=8, color='#3498db', marker='diamond', legend_label='Rewater')
    p1.legend.location = 'top_left'

    y_max = max(250, float(max(gs_dd.max(), gs_rw.max()) * 1.1))
    p2 = figure(title=f'gs Hysteresis \u2014 {title_suffix}',
               x_axis_label='\u03C8_xyl (MPa)', y_axis_label='gs (mmol/m\u00B2/s)',
               width=550, height=400, y_range=(0, y_max))
    p2.line(list(psix_dd_base), list(gs_dd_base), line_width=1.5, color='gray', alpha=0.5, legend_label='Baseline DD')
    p2.line(list(psix_rw_base), list(gs_rw_base), line_width=1.5, color='gray', alpha=0.5, line_dash='dashed', legend_label='Baseline RW')
    p2.line(list(psix_dd), list(gs_dd), line_width=2.5, color='red', legend_label='Drydown')
    p2.scatter(list(psix_dd), list(gs_dd), size=8, color='#e74c3c', marker='diamond', legend_label='Drydown')
    p2.line(list(psix_rw), list(gs_rw), line_width=2.5, color='blue', legend_label='Rewater')
    p2.scatter(list(psix_rw), list(gs_rw), size=8, color='#3498db', marker='diamond', legend_label='Rewater')
    p2.legend.location = 'top_right'

    return bokeh_row(p1, p2)

def make_dashboard(title, description, slider_specs, dashboard_name):
    """Build a Panel dashboard with sliders and a Compute button.
    
    slider_specs: list of tuples. Each tuple is either:
        (key, label, lo, hi, default, step)              — no unit conversion
        (key, label, lo, hi, default, step, scale_factor) — slider values are DIVIDED
            by scale_factor before passing to the ODE.
    """
    sliders = {}
    slider_widgets = []
    scale_factors = {}
    for spec in slider_specs:
        if len(spec) == 7:
            key, label, lo, hi, default, step_sz, sf = spec
            scale_factors[key] = sf
        else:
            key, label, lo, hi, default, step_sz = spec
            scale_factors[key] = 1.0
        s = pn.widgets.FloatSlider(name=label, start=lo, end=hi, value=default, step=step_sz, width=400)
        sliders[key] = s
        slider_widgets.append(s)

    # Status label and compute button
    status = pn.widgets.StaticText(value='Ready. Adjust sliders then click Compute.')
    compute_btn = pn.widgets.Button(name='Compute', button_type='success', width=200)
    reset_btn = pn.widgets.Button(name='Reset Defaults', button_type='warning', width=200)

    # Plot pane — starts empty
    plot_pane = pn.pane.Bokeh(make_hysteresis_plot(
        psix_dd_base, ABA_dd_base, gs_dd_base, psix_rw_base, ABA_rw_base, gs_rw_base,
        f'{dashboard_name} (baseline)'))

    def on_compute(event):
        status.value = 'Computing... (this takes ~30-60 seconds)'
        overrides = {}
        for k, s in sliders.items():
            overrides[k] = s.value / scale_factors[k]
        try:
            result = run_hysteresis(overrides)
            psix_dd, ABA_dd, gs_dd, psix_rw, ABA_rw, gs_rw = result
            plot_pane.object = make_hysteresis_plot(psix_dd, ABA_dd, gs_dd, psix_rw, ABA_rw, gs_rw, dashboard_name)
            status.value = 'Done!'
        except Exception as e:
            status.value = f'Error: {e}'

    def on_reset(event):
        for spec in slider_specs:
            key, default = spec[0], spec[4]
            sliders[key].value = default
        status.value = 'Sliders reset to defaults.'

    compute_btn.on_click(on_compute)
    reset_btn.on_click(on_reset)

    # Layout: two columns of sliders
    n = len(slider_widgets)
    mid = (n + 1) // 2
    col1 = pn.Column(*slider_widgets[:mid])
    col2 = pn.Column(*slider_widgets[mid:])
    slider_row = pn.Row(col1, col2)

    header = pn.pane.Markdown(f'### {title}\n{description}')
    buttons = pn.Row(compute_btn, reset_btn)

    return pn.Column(header, slider_row, buttons, status, plot_pane)

print('Dashboard builder ready (Panel + Bokeh).')

---
## Dashboard 1: Hydraulics

These parameters control **water transport** from xylem through mesophyll to guard cells, and the force balance that converts turgor to stomatal opening.

In [ ]:
dashboard_hydraulics = make_dashboard(
    title='Hydraulics Parameters',
    description='Adjust tissue resistances and stomatal conductance equation coefficients.',
    slider_specs=[
        # (key,    label,                      min,   max,   default, step)
        ('Rms',   'R_ms (MPa/mmol/m²/s)',         0.5,  20.0,   5.0,    0.5),
        ('Rsg',   'R_sg (MPa/mmol/m²/s)',         1.0,  30.0,  10.0,    0.5),
        ('gg1',   'g_g¹ (mmol/m²/s)',     0.5,  10.0,   2.9,    0.1),
        ('gg2',   'g_g² (mmol/m²/s)',            0.2,   3.0,   1.0,    0.1),
        ('ge1',   'g_e¹ (mmol/m²/s)',     0.5,  10.0,   2.9,    0.1),
        ('ge2',   'g_e² (mmol/m²/s)',            1.0,  30.0,  10.0,    0.5),
    ],
    dashboard_name='Hydraulics'
)
dashboard_hydraulics

---
## Dashboard 2: Stress Sensing (Vstress → ΔPg Curve)

These parameters shape the **stress-responsive ABA synthesis pathway**, modulating how guard cell turgor loss triggers ABA production.

In [ ]:
dashboard_vstress = make_dashboard(
    title='Vstress \u2192 \u0394Pg Curve',
    description='How guard cell turgor loss triggers stress-responsive ABA synthesis.',
    slider_specs=[
        ('vstress_exponent', 'Hill exponent n',     1.0,  6.0,  3.25,  0.25),
        ('vstress_khalf',    'K_half (MPa\u207f)',   0.2,  5.0,  1.4,   0.1),
    ],
    dashboard_name='Vstress Curve'
)
dashboard_vstress


---
## Dashboard 3: ABA Biosynthesis & Catabolism

These parameters control the **production and breakdown of ABA** via the autoregulation motif (SI Section S3.1, Eqs. S5, S8): stress-induced synthesis ($V_{stress}$), positive feedback ($V_+$, $K_+$), enzymatic degradation by A8H ($V_-$, $K_d$, $\gamma_A$), and A8H turnover ($K_-$, $\gamma_H$).

In [ ]:
dashboard_aba = make_dashboard(
    title='ABA Biosynthesis & Catabolism',
    description='Synthesis rates, feedback gains, and enzyme kinetics of the ABA autoregulatory motif.',
    slider_specs=[
        ('Vstress', 'V_stress (nM/s)',               1.0,   30.0,    7.37,   0.1),
        ('Vplus',   'V\u208a (nM/s)',              10.0,  200.0,   94.00,   1.0),
        ('Vminus',  'V\u208b (nM/s)',               0.1,   10.0,    1.51,   0.05),
        ('K_pm',    'K\u208a = K\u208b (nM)',    200.0, 5000.0, 1560.0,  20.0),
        ('K2',      'K_d (nM)',                    500.0, 5000.0, 2000.0,  50.0),
        ('kcat',    '\u03b3_A (1/s)',               0.05,    1.0,   0.25,   0.01),
        ('gammac',  '\u03b3_H (\u00d710\u207b\u00b3 1/s)', 0.5, 10.0, 3.072, 0.1, 1e3),
        ('n',       'n (Hill coefficient)',           1.0,    5.0,    3.0,   1.0),
    ],
    dashboard_name='ABA Biosynthesis'
)
dashboard_aba


---
## Dashboard 4: ABA Signaling Cascade

These parameters govern the **receptor-kinase-phosphatase cascade** (SI Section S3.2, Eqs. S28, S33) that translates cytosolic ABA into SLAC1 ion channel activation via the OST1 bicyclic futile cycle with non-competitive PP2C inhibition.

In [ ]:
dashboard_signaling = make_dashboard(
    title='ABA Signaling Cascade',
    description='Receptor binding, kinase/phosphatase rates, Michaelis constants, and protein levels.',
    slider_specs=[
        # Kinase / phosphatase rates
        # key    label                              min     max      default    step    [scale]
        ('k1',   'k₁ MAP3K→OST1 (1/s)',           0.005,   0.1,    0.03,     0.005),
        ('k2',   'k₂ PP2C→OST1 (1/s)',            0.005,   0.1,    0.033,    0.001),
        ('k3',   'k₃ OST1*→SLAC1 (1/s)',          0.5,    10.0,    3.75,     0.25),
        ('k4',   'k₄ PP2C→SLAC1 (1/s)',           0.02,    0.5,    0.139,    0.005),
        # Michaelis constants
        ('Km1',  'K_m1 OST1 phosph. (nM)',        500.0, 8000.0, 2500.0,   100.0),
        ('Km2',  'K_m2 OST1 dephosph. (nM)',       10.0,  500.0,   89.7,     5.0),
        ('Km3',  'K_m3 SLAC1 phosph. (µM)',         1.0,   50.0,   18.93,    0.5,  1e-3),
        ('Km4',  'K_m4 SLAC1 dephosph. (nM)',      50.0, 2000.0,  597.0,    10.0),
        # Receptor / inhibition
        ('KD',   'K_D ABA–PYR dissoc. (nM)',      200.0, 3000.0, 1000.0,    50.0),
        ('KI',   'K_I PP2C inhibition (nM)',         5.0,  100.0,   30.0,     1.0),
        # Protein totals
        ('OST1T',  '[OST1]_T (nM)',               200.0, 3000.0, 1000.0,   50.0),
        ('PP2CT',  '[PP2C]_T (nM)',               200.0, 3000.0, 1000.0,   50.0),
        ('SLAC1T', '[SLAC1]_T (nM)',              200.0, 3000.0, 1000.0,   50.0),
        ('MAP3KT',  '[MAP3K]_T (nM)',              200.0, 3000.0, 1000.0,   50.0),
    ],
    dashboard_name='ABA Signaling'
)
dashboard_signaling

---
## Dashboard 5: Electrophysiology

These parameters control **ion channels and pumps**, determining the electrical properties and ion fluxes in the guard cell.

In [ ]:
dashboard_electrophys = make_dashboard(
    title='Electrophysiology',
    description='Ion concentrations and ABA-pump sensitivity governing guard cell membrane potential.',
    slider_specs=[
        ('Clin0',           'Cl\u207b_in (mM)',    50.0, 500.0,  260.0,  10.0),
        ('Clout0',          'Cl\u207b_out (mM)',     5.0, 100.0,   22.0,   1.0),
        ('Kin0',            'K\u207a_in (mM)',     50.0, 500.0,  210.0,  10.0),
        ('Kout0',           'K\u207a_out (mM)',      5.0, 100.0,   20.0,   1.0),
        ('ABA_Hpump_Khalf', 'K_ABA (nM)',           10.0, 200.0,   50.0,   5.0),
        # --- GORK channel ---
        ('delta_GORK',       'δ_GORK',               0.5,   5.0,    2.0,   0.1),
        ('V1by2_Kconc_GORK', 'V½_GORK K⁺ (mM)',     10.0, 500.0,  150.0,  10.0),
        # --- KAT1 channel ---
        ('delta_KAT1',       'δ_KAT1',               0.5,   5.0,    1.6,   0.1),
        ('V1by2_KAT1',       'V½_KAT1 (mV)',       -300.0, -50.0, -170.0,   5.0, 1000.0),
        # --- H⁺-ATPase pump ---
        ('DeltaGATP',        'ΔG_ATP (kJ/mol)',     -40.0, -15.0,  -26.0,   0.5, 0.001),
        ('E0pump',           'E₀_pump (×10⁶)',        0.1,   5.0,  1.1033,  0.0001, 1e-6),
        ('k1pump',           'k₁_pump',              0.01,   2.0,   0.55,  0.01),
        ('k2pump',           'k₂_pump (×10⁻⁴)',      0.1,  10.0,   2.58,   0.01, 1e4),
        # --- Symporter / SLAC1 scaling ---
        ('V0Cl',             'V₀_Cl (×10⁴)',         1.0,  30.0,   9.25,  0.25, 1e-4),
        # --- ABA pump inhibition ---
        ('ABA_Hpump_n',      'n_ABA pump',            0.5,   4.0,    1.0,   0.5),
    ],
    dashboard_name='Electrophysiology'
)
dashboard_electrophys


In [ ]:
# Build the page that will be served by panel convert.
pn.Column(
    "# Stomatal hysteresis dashboard",
    "Drydown / rewatering hysteresis cycle (ψ_xyl steps).",
    pn.Tabs(
        ("Hydraulics",        dashboard_hydraulics),
        ("V_stress",          dashboard_vstress),
        ("ABA biosynthesis",  dashboard_aba),
        ("ABA signaling",     dashboard_signaling),
        ("Electrophysiology", dashboard_electrophys),
    ),
).servable()
